# 05.05 · Cost, latency, caching, retries

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Count tokens before sending; compute cost from `usage`
- [ ] Handle failures: timeouts, 429 rate limits, 5xx — retries with exponential backoff + jitter
- [ ] Run many calls concurrently with a concurrency limit
- [ ] Cache: your own response cache vs provider prompt caching; batch APIs
- [ ] Log every call (model, tokens, latency, cost) — the seed of observability (chapter 11)

## Resources
- [OpenAI – Prompt caching](https://platform.openai.com/docs/guides/prompt-caching) · [Anthropic – Prompt caching](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching) — free
- [OpenAI – Batch API](https://platform.openai.com/docs/guides/batch) · [Rate limits](https://platform.openai.com/docs/guides/rate-limits) — free
- [AWS – Exponential backoff and jitter](https://aws.amazon.com/blogs/architecture/exponential-backoff-and-jitter/) — free

In [ ]:
import asyncio
import csv
import hashlib
import json
import random
import shelve
import time
from datetime import datetime, timezone

import openai
import tiktoken
from openai import AsyncOpenAI

from aieng import DATA
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. Count tokens before you send
Use the model's own tokenizer when you can; `tiktoken` (o200k) is a good approximation for budgeting.
The **authoritative** count is the `usage` field the API returns.

In [ ]:
enc = tiktoken.get_encoding("o200k_base")
prompt = "Summarise the plot of Don Quixote in three sentences."
r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], **EXTRA)
print("tiktoken estimate:", len(enc.encode(prompt)), "| API usage.prompt_tokens:", r.usage.prompt_tokens,
      "(includes chat-template tokens)")

## 2. Cost tracking
Keep prices in one place (per 1M tokens). The values below are placeholders: set real ones for the models you use.

In [ ]:
PRICES = {  # model -> ($/1M input, $/1M output). Placeholders: update from pricing pages.
    MODEL: (0.0, 0.0),
    "gemini-flash-latest": (0.30, 2.50),
    "gpt-5-mini": (0.25, 2.00),
}

def cost_usd(model, usage):
    p_in, p_out = PRICES.get(model, (0.0, 0.0))
    return (usage.prompt_tokens * p_in + usage.completion_tokens * p_out) / 1e6

print(f"${cost_usd(MODEL, r.usage):.6f}")

## 3. Failures and retries
| Error | Retry? |
|---|---|
| timeout, connection error | yes |
| 429 rate limit | yes, with backoff (respect `Retry-After` if given) |
| 500/502/503/529 overloaded | yes |
| 400 bad request, 401 auth, 404 model not found | **no** — fix your code/config |

The `openai` SDK already retries some errors (`max_retries=2` by default) and has `timeout=`. Configure them explicitly.
For anything else (or other SDKs), write the loop: **exponential backoff with jitter** so many clients don't retry in sync.

In [ ]:
RETRYABLE = (openai.APITimeoutError, openai.APIConnectionError, openai.RateLimitError, openai.InternalServerError)


def with_retries(fn, max_attempts=5, base=0.5, cap=20.0):
    for attempt in range(max_attempts):
        try:
            return fn()
        except RETRYABLE as e:
            if attempt == max_attempts - 1:
                raise
            sleep = random.uniform(0, min(cap, base * 2**attempt))  # "full jitter"
            print(f"  {type(e).__name__}, retry {attempt + 1} in {sleep:.2f}s")
            time.sleep(sleep)


# Simulate a flaky provider: a client pointed at a port where nothing listens -> APIConnectionError
bad = openai.OpenAI(base_url="http://localhost:9/v1", api_key="x", max_retries=0, timeout=1)
try:
    with_retries(lambda: bad.chat.completions.create(model="x", messages=[{"role": "user", "content": "hi"}]), max_attempts=3, base=0.2)
except openai.APIConnectionError:
    print("gave up after retries (correct: the server doesn't exist)")

## 4. Concurrency with a limit
`AsyncOpenAI` + `asyncio.Semaphore` (01.01). Set the limit below your provider's rate limit.
Local Ollama processes few requests in parallel by default (`OLLAMA_NUM_PARALLEL`), so gains are smaller locally than on APIs.

In [ ]:
ac = AsyncOpenAI(base_url=str(c.base_url), api_key=c.api_key, max_retries=3, timeout=120)
questions = [f"In one sentence, what is {t}?" for t in ["recursion", "a mutex", "TCP", "DNS", "a hash function", "Big-O notation"]]


async def ask(q, sem):
    async with sem:
        r = await ac.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": q}], **EXTRA)
        return r.choices[0].message.content


for limit in [1, 4]:
    sem = asyncio.Semaphore(limit)  # ONE semaphore shared by all calls; a new one per call would mean no limit
    start = time.perf_counter()
    answers = await asyncio.gather(*(ask(q, sem) for q in questions))
    print(f"concurrency {limit}: {time.perf_counter() - start:.1f}s for {len(questions)} calls")
print(answers[0])

Locally the gain is small (one laptop GPU); against hosted APIs concurrency 8–16 is often ~8–16× faster. Exercise 1 measures it.

## 5. Your own response cache
Identical request → identical answer (at temperature 0). Cache by a hash of the full request. Huge savings in dev loops and evals.

In [ ]:
CACHE_PATH = str(DATA / "llm_cache")


def cached_completion(**request):
    key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()
    with shelve.open(CACHE_PATH) as db:
        if key in db:
            return db[key], True
        r = c.chat.completions.create(**request)
        db[key] = r.choices[0].message.content
        return db[key], False


req = dict(model=MODEL, messages=[{"role": "user", "content": "Name three sorting algorithms."}], temperature=0, **EXTRA)
for _ in range(2):
    start = time.perf_counter()
    text, hit = cached_completion(**req)
    print(f"cache hit={hit} in {(time.perf_counter() - start) * 1000:.0f} ms")

## 6. Provider-side prompt caching & batch APIs
- **Prompt caching**: providers cache the processed *prefix* of your prompt. Repeated prefixes (long system prompt, documents, tool
  definitions) cost much less and start faster. OpenAI does it automatically for long prefixes; Anthropic uses explicit
  `cache_control` breakpoints; Gemini has implicit + explicit context caching. **Design prompts with the static part first, variable part last.**
- **Batch APIs**: submit thousands of requests as a file, get results within ~24h at a large discount (often ~50%). Perfect for evals,
  dataset labelling, offline summarisation.

## 7. Log every call
Minimal call log to CSV. Chapter 11 replaces this with proper tracing (Langfuse etc.), but the fields are the same.

In [ ]:
LOG = DATA / "llm_calls.csv"


def logged_chat(messages, model=MODEL, **kwargs):
    start = time.perf_counter()
    r = with_retries(lambda: c.chat.completions.create(model=model, messages=messages, **{**EXTRA, **kwargs}))
    row = {"ts": datetime.now(timezone.utc).isoformat(), "model": model, "in_tokens": r.usage.prompt_tokens,
           "out_tokens": r.usage.completion_tokens, "latency_s": round(time.perf_counter() - start, 3),
           "cost_usd": cost_usd(model, r.usage), "finish": r.choices[0].finish_reason}
    new = not LOG.exists()
    with LOG.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=row.keys())
        if new:
            w.writeheader()
        w.writerow(row)
    return r.choices[0].message.content


logged_chat([{"role": "user", "content": "What's a good name for a cat?"}])
print(LOG.read_text().splitlines()[-2:])

## Exercises — toward Milestone M2
1. Measure concurrency 1, 2, 4, 8 against a hosted provider (Groq/Gemini). Where do you start hitting 429s?
2. Wrap your 05.02 `Chat` class with retries, a timeout, cost logging and a **per-session budget** (refuse when over $X).
3. Design a prompt for a 2,000-token system prompt + user question so that provider caching applies. With an API key, measure
   `usage.prompt_tokens_details.cached_tokens` on the second call.

In [ ]:
# your code here